# Einfluss der Semantik auf Merkmalsextraktoren und Matcher (Bachelorarbeit)

Evaluierungspipeline für 6 Feature-Detektoren (SIFT, ORB, BRISK, SuperPoint, DISK, KeyNetHardNet) und 5 Matching-Verfahren (BruteForce, FLANN, KNN+Lowe, FAISS, CrossCheck) auf dem HPatches-Datensatz, mit semantischer Filterung der Matches über COCO-Annotationen.

**Voraussetzungen:** Datensätze nach `data/` legen (siehe README), Pakete aus `requirements.txt` installieren. Pfade lassen sich über die Umgebungsvariablen `DATA_DIR` und `OUTPUT_DIR` ändern.


In [ ]:
#python -m pip install --upgrade pip

#pip install notebook ipykernel ipywidgets

#pip install numpy pandas matplotlib seaborn tqdm openpyxl

#pip install opencv-contrib-python

#pip install kornia faiss-cpu

#pip install git+https://github.com/cvg/LightGlue.git

#pip uninstall torch torchvision torchaudio -y

#pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

In [ ]:
import os

# Konfiguration: Pfade über Umgebungsvariablen anpassbar
DATA_DIR = os.environ.get('DATA_DIR', 'data')        # enthält hpatches-sequences-release/ und hpatches-yolo-jpg/
OUTPUT_DIR = os.environ.get('OUTPUT_DIR', 'outputs') # Ergebnisse (CSV, PNG, Caches)
os.makedirs(OUTPUT_DIR, exist_ok=True)

DATASET_PATH = os.path.join(DATA_DIR, 'hpatches-sequences-release')

if os.path.exists(DATASET_PATH):
    scenes = sorted([
        d for d in os.listdir(DATASET_PATH)
        if os.path.isdir(os.path.join(DATASET_PATH, d))
    ])
    print(f'Dataset gefunden')
    print(f'Anzahl Szenen: {len(scenes)}')
    print(f'Illumination: {sum(1 for s in scenes if s.startswith("i_"))}')
    print(f'Viewpoint:    {sum(1 for s in scenes if s.startswith("v_"))}')
else:
    print('Pfad nicht gefunden')
    print('Bitte HPatches herunterladen und DATA_DIR setzen (siehe README).')

In [ ]:
import pickle
import cv2
import numpy as np
import torch
import kornia.feature as KF
import faiss
import time
import random
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from pathlib import Path
from lightglue import SuperPoint as SuperPointExtractor

DEVICE = torch.device('cpu')
print(f'Gerät: {DEVICE}')

IMG_SIZE      = (480, 640)
LOWE_RATIO    = 0.75
RANSAC_THRESH = 3.0
THRESH_ILLUMINATION = 3.0
THRESH_VIEWPOINT    = 30.0
N_RANDOM_PAIRS      = 2

In [ ]:
def load_image(path):
    img = cv2.imread(str(path))
    if img is None:
        return None
    img = cv2.resize(img, (IMG_SIZE[1], IMG_SIZE[0]))
    return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

def img_to_tensor(img_gray):
    t = torch.tensor(img_gray, dtype=torch.float32) / 255.0
    return t.unsqueeze(0).unsqueeze(0).to(DEVICE)

def load_homography(scene_path, img_idx):
    h_file = os.path.join(scene_path, f'H_1_{img_idx}')
    if not os.path.exists(h_file):
        return None
    return np.loadtxt(h_file)

In [ ]:
#CPU OpenCV
_orb_cv   = cv2.ORB_create(nfeatures=50000)
_brisk_cv = cv2.BRISK_create()
_sift_cv  = cv2.SIFT_create(nfeatures=0)

def detect_sift(img_gray):
    kps, descs = _sift_cv.detectAndCompute(img_gray, None)
    return kps, descs

def detect_orb(img_gray):
    kps, descs = _orb_cv.detectAndCompute(img_gray, None)
    return kps, descs

def detect_brisk(img_gray):
    kps, descs = _brisk_cv.detectAndCompute(img_gray, None)
    return kps, descs

In [ ]:
print('Lade CNN Modelle auf CPU')

_superpoint = SuperPointExtractor(
    max_num_keypoints=None
).eval().to(DEVICE)

_disk = KF.DISK.from_pretrained('depth').eval().to(DEVICE)

_keynet_hardnet = KF.KeyNetHardNet(
    num_features=8000,
    device=DEVICE
).eval()

print(f'Alle 3 CNN Modelle auf {DEVICE} geladen')

In [ ]:
def detect_superpoint(img_gray):
    with torch.no_grad():
        output = _superpoint({'image': img_to_tensor(img_gray)})
    kps_xy = output['keypoints'][0].cpu().numpy()
    descs  = output['descriptors'][0].cpu().numpy()
    kps    = [cv2.KeyPoint(float(x), float(y), 1.0) for x, y in kps_xy]
    return kps, descs.astype(np.float32)

def detect_disk(img_gray):
    with torch.no_grad():
        inp      = img_to_tensor(img_gray).expand(-1, 3, -1, -1)
        features = _disk(inp, 8000, pad_if_not_divisible=True)
    kps_xy = features[0].keypoints.cpu().numpy()
    descs  = features[0].descriptors.cpu().numpy()
    kps    = [cv2.KeyPoint(float(x), float(y), 1.0) for x, y in kps_xy]
    return kps, descs.astype(np.float32)

def detect_keynethardnet(img_gray):
    with torch.no_grad():
        inp                 = img_to_tensor(img_gray)
        lafs, scores, descs = _keynet_hardnet(inp)
    kps_xy = KF.get_laf_center(lafs)[0].cpu().numpy()
    descs  = descs[0].cpu().numpy()
    kps    = [cv2.KeyPoint(float(x), float(y), 1.0) for x, y in kps_xy]
    return kps, descs.astype(np.float32)

print('CNN Detektoren bereit (CPU)')

In [ ]:
DETECTORS = {
    'SIFT':          detect_sift,
    'ORB':           detect_orb,
    'BRISK':         detect_brisk,
    'SuperPoint':    detect_superpoint,
    'DISK':          detect_disk,
    'KeyNetHardNet': detect_keynethardnet,
}

BINARY_DETECTORS = {'ORB', 'BRISK'}

print(f'{"Detektor":<15} {"Hardware"}')
print('-' * 30)
for det in DETECTORS.keys():
    print(f'{det:<15} CPU')


In [ ]:
def match_bf(desc1, desc2, binary=False):
    norm    = cv2.NORM_HAMMING if binary else cv2.NORM_L2
    matcher = cv2.BFMatcher(norm, crossCheck=False)
    matches = matcher.match(desc1, desc2)
    if len(matches) > 0:
        avg_dist = sum(m.distance for m in matches) / len(matches)
        matches  = [m for m in matches if m.distance < 0.75 * avg_dist]
    return sorted(matches, key=lambda x: x.distance)

def match_flann(desc1, desc2, binary=False):
    if binary:
        index_params  = dict(algorithm=6, table_number=6,
                             key_size=12, multi_probe_level=1)
    else:
        index_params  = dict(algorithm=1, trees=5)
    search_params = dict(checks=50)
    matcher = cv2.FlannBasedMatcher(index_params, search_params)
    matches = matcher.match(desc1, desc2)
    if len(matches) > 0:
        avg_dist = sum(m.distance for m in matches) / len(matches)
        matches  = [m for m in matches if m.distance < 0.75 * avg_dist]
    return sorted(matches, key=lambda x: x.distance)

def match_knn_lowe(desc1, desc2, binary=False, ratio=0.75):
    if desc1 is None or desc2 is None:
        return []
    if len(desc1) < 2 or len(desc2) < 2:
        return []
    norm    = cv2.NORM_HAMMING if binary else cv2.NORM_L2
    matcher = cv2.BFMatcher(norm)
    knn     = matcher.knnMatch(desc1, desc2, k=2)
    good    = []
    for pair in knn:
        if len(pair) == 2:
            m, n = pair
            if m.distance < ratio * n.distance:
                good.append(m)
    return good

def match_faiss(desc1, desc2, binary=False):
    if binary:
        return match_bf(desc1, desc2, binary=True)
    d1 = desc1.astype(np.float32).copy()
    d2 = desc2.astype(np.float32).copy()
    faiss.normalize_L2(d1)
    faiss.normalize_L2(d2)
    dim   = d1.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(d2)
    distances, I = index.search(d1, 1)
    matches = []
    for i in range(len(d1)):
        if I[i][0] >= 0:
            matches.append(cv2.DMatch(i, int(I[i][0]),
                                      float(distances[i][0])))
    if len(matches) > 0:
        matches = sorted(matches, key=lambda x: x.distance, reverse=True)
        matches = matches[:int(len(matches) * 0.75)]
    return matches

def match_crosscheck(desc1, desc2, binary=False):
    if desc1 is None or desc2 is None:
        return []
    norm    = cv2.NORM_HAMMING if binary else cv2.NORM_L2
    matcher = cv2.BFMatcher(norm, crossCheck=True)
    matches = matcher.match(desc1, desc2)
    return sorted(matches, key=lambda x: x.distance)

MATCHERS = {
    'BruteForce': match_bf,
    'FLANN':      match_flann,
    'KNN+Lowe':   match_knn_lowe,
    'FAISS':      match_faiss,
    'CrossCheck': match_crosscheck,
}

print(f'{"Matcher":<12} {"Hardware"}')
print('-' * 25)
for mat in MATCHERS.keys():
    print(f'{mat:<12} CPU')


In [ ]:
def compute_precision(kps1, kps2, matches, H_gt,
                      scene_name='', threshold=None):
    if len(matches) == 0 or H_gt is None:
        return 0.0, 0.0
    if threshold is None:
        threshold = THRESH_VIEWPOINT if scene_name.startswith('v_') \
                    else THRESH_ILLUMINATION
    pts1      = np.float32([kps1[m.queryIdx].pt
                            for m in matches]).reshape(-1,1,2)
    pts2      = np.float32([kps2[m.trainIdx].pt
                            for m in matches]).reshape(-1,1,2)
    pts1_proj = cv2.perspectiveTransform(pts1, H_gt).reshape(-1,2)
    pts2_flat = pts2.reshape(-1,2)
    errors    = np.linalg.norm(pts1_proj - pts2_flat, axis=1)
    correct     = np.sum(errors < threshold)
    praezision  = float(correct / len(errors))
    korrekt_pct = praezision * 100
    return praezision, korrekt_pct

def compute_homography(kps1, kps2, matches):
    if len(matches) < 4:
        return 0.0
    pts1 = np.float32([kps1[m.queryIdx].pt for m in matches])
    pts2 = np.float32([kps2[m.trainIdx].pt for m in matches])
    _, mask = cv2.findHomography(pts1, pts2, cv2.RANSAC, RANSAC_THRESH)
    if mask is None:
        return 0.0
    return float(mask.ravel().sum() / len(mask))


In [ ]:

ALL_SCENES = scenes
extraction_results = []

extraction_cache = {}

print(f'Starte Extraction auf CPU')
print(f'{len(ALL_SCENES)} Szenen x 6 Bilder = {len(ALL_SCENES)*6} Bilder\n')
print(f'{"Detektor":<15} {"Bilder":>8} {"Ø Keypoints":>12} {"Ø Zeit (ms)":>12} {"Min KP":>8} {"Max KP":>8}')
print('-' * 68)

for det_name, det_fn in DETECTORS.items():
    det_results = []
    extraction_cache[det_name] = {}

    for scene in tqdm(ALL_SCENES, desc=det_name, leave=False):
        for img_idx in range(1, 7):
            img_path = Path(DATASET_PATH) / scene / f'{img_idx}.ppm'
            img      = load_image(img_path)
            if img is None:
                continue

            t0 = time.time()
            try:
                kps, descs = det_fn(img)
                n_kps      = len(kps) if kps else 0
            except Exception as e:
                print(f'Fehler: {det_name}/{scene}/{img_idx}: {e}')
                kps, descs, n_kps = [], None, 0
            ms = (time.time() - t0) * 1000


            extraction_cache[det_name][(scene, img_idx)] = (kps, descs, ms)

            #Tabell
            det_results.append({
                'Detektor':  det_name,
                'Typ':       'CNN' if det_name in
                             ['SuperPoint','DISK','KeyNetHardNet']
                             else 'Klassisch',
                'Szene':     scene,
                'Szenentyp': 'Illumination' if scene.startswith('i_')
                             else 'Viewpoint',
                'Bild':      img_idx,
                'Keypoints': n_kps,
                'Zeit_ms':   round(ms, 2),
            })

    extraction_results.extend(det_results)


    n       = len(det_results)
    avg_kps = sum(r['Keypoints'] for r in det_results) / n
    min_kps = min(r['Keypoints'] for r in det_results)
    max_kps = max(r['Keypoints'] for r in det_results)
    avg_ms  = sum(r['Zeit_ms']   for r in det_results) / n
    print(f'{det_name:<15} {n:>8} {avg_kps:>12.0f} {avg_ms:>12.1f} {min_kps:>8} {max_kps:>8}')

print('-' * 68)
print(f'Gesamt: {len(extraction_results)} Messungen')

df_extract = pd.DataFrame(extraction_results)
df_extract.to_csv(os.path.join(OUTPUT_DIR, 'ganz_bild_extraction_cpu.csv'), index=False)

# Tabell anzeigen
print('\n')
print('=' * 72)
print('  TABELLE – Feature Extraction (alle 116 Szenen x 6 Bilder)')
print('=' * 72)

tabelle_extract = df_extract.groupby('Detektor').agg(
    Typ           = ('Typ',       'first'),
    Bilder        = ('Bild',      'count'),
    Ø_Keypoints   = ('Keypoints', 'mean'),
    Ø_Zeit_ms     = ('Zeit_ms',   'mean'),

).round(1)

det_order = ['SIFT','ORB','BRISK','SuperPoint','DISK','KeyNetHardNet']
tabelle_extract = tabelle_extract.reindex(det_order)
print(tabelle_extract.to_string())
print('=' * 72)

print(f'\nCache enthaelt {sum(len(v) for v in extraction_cache.values())} Einträge')


In [ ]:
print('Speichere ganz_bild_extraction_cache.pkl')

def keypoint_to_tuple(kp):
    return (
        kp.pt[0],
        kp.pt[1],
        kp.size,
        kp.angle,
        kp.response,
        kp.octave,
        kp.class_id
    )

serializable_cache = {}

for det_name, det_cache in extraction_cache.items():
    serializable_cache[det_name] = {}

    for key, value in det_cache.items():
        kps, descs, ms = value

        kps_serialized = [keypoint_to_tuple(kp) for kp in kps] if kps else []

        serializable_cache[det_name][key] = (
            kps_serialized,
            descs,
            ms
        )

with open(os.path.join(OUTPUT_DIR, 'ganz_bild_extraction_cache.pkl'), 'wb') as f:
    pickle.dump(list(serializable_cache.keys()), f)
    for det_name in serializable_cache:
        pickle.dump(det_name, f)
        pickle.dump(serializable_cache[det_name], f)

print(f'Cache gespeichert: {list(serializable_cache.keys())}')

In [ ]:
import pickle
import numpy as np
import pandas as pd
import cv2

def tuple_to_keypoint(t):
    x, y, size, angle, response, octave, class_id = t
    kp = cv2.KeyPoint(
        x=float(x),
        y=float(y),
        size=float(size),
        angle=float(angle),
        response=float(response),
        octave=int(octave),
        class_id=int(class_id)
    )
    return kp

print(' Lade ganz_bild_extraction_cache.pkl')

with open(os.path.join(OUTPUT_DIR, 'ganz_bild_extraction_cache.pkl'), 'rb') as f:
    det_list = pickle.load(f)
    extraction_cache = {}

    for _ in det_list:
        det_name = pickle.load(f)
        det_cache = pickle.load(f)

        extraction_cache[det_name] = {}

        for key, value in det_cache.items():
            kps_serialized, descs, ms = value
            kps = [tuple_to_keypoint(t) for t in kps_serialized] if kps_serialized else []
            extraction_cache[det_name][key] = (kps, descs, ms)

print(f' Cache geladen: {list(extraction_cache.keys())}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

det_order = ['SIFT', 'ORB', 'BRISK', 'SuperPoint', 'DISK', 'KeyNetHardNet']

avg_kps_list = []
avg_time_list = []

for det in det_order:
    if det not in extraction_cache:
        avg_kps_list.append(0)
        avg_time_list.append(0)
        continue

    cache = extraction_cache[det]

    kps_counts = []
    times = []

    for (scene, img_idx), (kps, desc, ms) in cache.items():
        kps_counts.append(len(kps) if kps else 0)
        times.append(ms)

    avg_kps = np.mean(kps_counts) if len(kps_counts) > 0 else 0
    avg_ms  = np.mean(times) if len(times) > 0 else 0

    avg_kps_list.append(avg_kps)
    avg_time_list.append(avg_ms)

# Diagramm Keypoints
fig, ax = plt.subplots(figsize=(12, 7))

x = np.arange(len(det_order))
bars = ax.bar(x, avg_kps_list)

for bar, val in zip(bars, avg_kps_list):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height(),
        f'{val:.0f}',
        ha='center',
        va='bottom',
        fontsize=14,
        fontweight='bold',
        color='black'
    )

ax.set_xticks(x)
ax.set_xticklabels(det_order)
ax.set_title('Durchschnittliche Anzahl Keypoints pro Detektor')
ax.set_ylabel('Ø Keypoints')
ax.set_xlabel('Extraktionsmethoden')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'ganz_bild_extraction_keypoints.png'), dpi=300)
plt.show()

# Diagramm Zeit
fig, ax = plt.subplots(figsize=(12, 7))

bars = ax.bar(x, avg_time_list)

for bar, val in zip(bars, avg_time_list):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height(),
        f'{val:.1f}',
        ha='center',
        va='bottom',
        fontsize=14,
        fontweight='bold',
        color='black'
    )

ax.set_xticks(x)
ax.set_xticklabels(det_order)
ax.set_title('Durchschnittliche Extraktionszeit pro Detektor')
ax.set_ylabel('Zeit (ms)')
ax.set_xlabel('Extraktionsmethoden')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'ganz_bild_extraction_time.png'), dpi=300)
plt.show()

In [ ]:
# GPU Extraction

import os
import time
import pickle
import torch
import cv2
import numpy as np
import kornia.feature as KF
from pathlib import Path
from tqdm import tqdm
from lightglue import SuperPoint as SuperPointExtractor


if 'scenes' not in globals():
    raise RuntimeError("Variable 'scenes' nicht gefunden.")

if 'DATASET_PATH' not in globals():
    raise RuntimeError("Variable 'DATASET_PATH' nicht gefunden.")

if 'load_image' not in globals():
    raise RuntimeError("Funktion 'load_image' nicht gefunden.")


ALL_SCENES = scenes

OUTPUT_DIR = os.environ.get('OUTPUT_DIR', 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)


# GPU prüfen
DEVICE_GPU = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'GPU-Gerät: {DEVICE_GPU}')

if DEVICE_GPU.type != 'cuda':
    print("Keine CUDA-GPU gefunden.")

else:

    def img_to_tensor_gpu(img_gray):
        t = torch.tensor(img_gray, dtype=torch.float32) / 255.0
        return t.unsqueeze(0).unsqueeze(0).to(DEVICE_GPU)


    def kps_to_list(kps):
        if not kps:
            return []
        return [
            (
                float(kp.pt[0]),
                float(kp.pt[1]),
                float(kp.size),
                float(kp.angle),
                float(kp.response),
                int(kp.octave),
                int(kp.class_id)
            )
            for kp in kps
        ]


    print('Lade CNN-Modelle auf GPU')

    _superpoint_gpu = SuperPointExtractor(
        max_num_keypoints=None
    ).eval().to(DEVICE_GPU)

    _disk_gpu = KF.DISK.from_pretrained('depth').eval().to(DEVICE_GPU)

    _keynet_hardnet_gpu = KF.KeyNetHardNet(
        num_features=8000,
        device=DEVICE_GPU
    ).eval()

    print(f'Alle 3 CNN-Modelle auf {DEVICE_GPU} geladen')


    def detect_superpoint_gpu(img_gray):
        with torch.no_grad():
            output = _superpoint_gpu({'image': img_to_tensor_gpu(img_gray)})
        kps_xy = output['keypoints'][0].detach().cpu().numpy()
        descs  = output['descriptors'][0].detach().cpu().numpy()
        kps    = [cv2.KeyPoint(float(x), float(y), 1.0) for x, y in kps_xy]
        return kps, descs.astype(np.float32)


    def detect_disk_gpu(img_gray):
        with torch.no_grad():
            inp = img_to_tensor_gpu(img_gray).expand(-1, 3, -1, -1)
            features = _disk_gpu(inp, 8000, pad_if_not_divisible=True)
        kps_xy = features[0].keypoints.detach().cpu().numpy()
        descs  = features[0].descriptors.detach().cpu().numpy()
        kps    = [cv2.KeyPoint(float(x), float(y), 1.0) for x, y in kps_xy]
        return kps, descs.astype(np.float32)


    def detect_keynethardnet_gpu(img_gray):
        with torch.no_grad():
            inp = img_to_tensor_gpu(img_gray)
            lafs, scores, descs = _keynet_hardnet_gpu(inp)
        kps_xy = KF.get_laf_center(lafs)[0].detach().cpu().numpy()
        descs  = descs[0].detach().cpu().numpy()
        kps    = [cv2.KeyPoint(float(x), float(y), 1.0) for x, y in kps_xy]
        return kps, descs.astype(np.float32)


    DETECTORS_GPU = {
        'SuperPoint_GPU': detect_superpoint_gpu,
        'DISK_GPU': detect_disk_gpu,
        'KeyNetHardNet_GPU': detect_keynethardnet_gpu,
    }


    # GPU Extraction
    gpu_extraction_cache = {}

    print(f'\nStarte GPU-Extraction')
    print(f'{len(ALL_SCENES)} Szenen x 6 Bilder = {len(ALL_SCENES) * 6} Bilder\n')
    print(f'{"Detektor":<20} {"Bilder":>8} {"Ø Keypoints":>12} {"Ø Zeit (ms)":>12} {"Min KP":>8} {"Max KP":>8}')
    print('-' * 76)

    for det_name, det_fn in DETECTORS_GPU.items():
        gpu_extraction_cache[det_name] = {}
        kp_list = []
        ms_list = []

        for scene in tqdm(ALL_SCENES, desc=det_name, leave=False):
            for img_idx in range(1, 7):
                img_path = Path(DATASET_PATH) / scene / f'{img_idx}.ppm'
                img = load_image(img_path)

                if img is None:
                    continue

                torch.cuda.synchronize()
                t0 = time.perf_counter()

                try:
                    kps, descs = det_fn(img)
                    n_kps = len(kps) if kps else 0
                except Exception as e:
                    print(f'Fehler: {det_name}/{scene}/{img_idx}: {e}')
                    kps, descs, n_kps = [], None, 0

                torch.cuda.synchronize()
                ms = (time.perf_counter() - t0) * 1000

                gpu_extraction_cache[det_name][(scene, img_idx)] = (
                    kps_to_list(kps),
                    descs,
                    ms
                )

                kp_list.append(n_kps)
                ms_list.append(ms)

        print(
            f'{det_name:<20} '
            f'{len(kp_list):>8} '
            f'{np.mean(kp_list):>12.0f} '
            f'{np.mean(ms_list):>12.1f} '
            f'{np.min(kp_list):>8} '
            f'{np.max(kp_list):>8}'
        )

    print('-' * 76)
    print('GPU-Extraction fertig')


    # Cache lokal speichern
    gpu_cache_path = os.path.join(OUTPUT_DIR, 'ganz_bild_extraction_gpu_cache.pkl')

    with open(gpu_cache_path, 'wb') as f:
        pickle.dump(gpu_extraction_cache, f)

    print(f'GPU-Cache gespeichert unter:')
    print(gpu_cache_path)

In [ ]:
import os
import pandas as pd
import numpy as np

print("Speichere GPU-Extraction-Ergebnisse als CSV")

if 'gpu_extraction_cache' not in globals():
    raise RuntimeError(
        "gpu_extraction_cache nicht gefunden. "
    )

rows = []

for det_name, cache in gpu_extraction_cache.items():
    det_clean = det_name.replace('_GPU', '')

    for (scene, img_idx), (kps, descs, ms) in cache.items():
        n_kps = len(kps) if kps is not None else 0

        rows.append({
            'Detektor': det_clean,
            'Hardware': 'GPU',
            'Scene': scene,
            'Image_Index': img_idx,
            'Keypoints': n_kps,
            'Time_ms': ms
        })

df_gpu = pd.DataFrame(rows)

OUTPUT_DIR = os.environ.get('OUTPUT_DIR', 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

gpu_csv_path = os.path.join(OUTPUT_DIR, 'ganz_bild_extraction_gpu.csv')

df_gpu.to_csv(gpu_csv_path, index=False)

print("GPU-CSV gespeichert:", gpu_csv_path)
print("Anzahl Zeilen:", len(df_gpu))

if len(df_gpu) > 0:
    print("Detektoren:", sorted(df_gpu['Detektor'].unique()))
    print("Spalten:", df_gpu.columns.tolist())
    display(df_gpu.head())
else:
    print("Keine GPU-Daten vorhanden.")

In [ ]:
import pandas as pd
import os


cpu_csv_path = os.path.join(OUTPUT_DIR, 'ganz_bild_extraction_cpu.csv')
gpu_csv_path = os.path.join(OUTPUT_DIR, 'ganz_bild_extraction_gpu.csv')

combined_csv_path = os.path.join(OUTPUT_DIR, 'ganz_bild_extraction_cpu_gpu_combined.csv')


if not os.path.exists(cpu_csv_path):
    raise FileNotFoundError(f"CPU-Datei nicht gefunden: {cpu_csv_path}")

if not os.path.exists(gpu_csv_path):
    raise FileNotFoundError(f"GPU-Datei nicht gefunden: {gpu_csv_path}")

print("Lade CPU-Datei:", cpu_csv_path)
df_cpu = pd.read_csv(cpu_csv_path)

print("Lade GPU-Datei:", gpu_csv_path)
df_gpu = pd.read_csv(gpu_csv_path)

print("\nCPU-Spalten:", df_cpu.columns.tolist())
print("GPU-Spalten:", df_gpu.columns.tolist())


def standardize_extraction_df(df, hardware_name):
    df = df.copy()

    det_cols = ['Detektor', 'detector', 'Detector', 'det_name', 'Method', 'method']
    det_col = None
    for col in det_cols:
        if col in df.columns:
            det_col = col
            break

    kp_cols = ['Keypoints', 'keypoints', 'num_keypoints', 'kps', 'Anzahl_Keypoints']
    kp_col = None
    for col in kp_cols:
        if col in df.columns:
            kp_col = col
            break

    time_cols = ['Time_ms', 'time_ms', 'Zeit_ms', 'ms', 'Time', 'Zeit']
    time_col = None
    for col in time_cols:
        if col in df.columns:
            time_col = col
            break

    scene_cols = ['Scene', 'scene', 'Szene']
    scene_col = None
    for col in scene_cols:
        if col in df.columns:
            scene_col = col
            break

    img_cols = ['Image_Index', 'image_index', 'Img_Index', 'img_idx', 'Bild_Index', 'Bild']
    img_col = None
    for col in img_cols:
        if col in df.columns:
            img_col = col
            break

    if det_col is None:
        raise ValueError(f"Keine Detektor-Spalte gefunden. Spalten: {df.columns.tolist()}")

    if kp_col is None:
        raise ValueError(f"Keine Keypoints-Spalte gefunden. Spalten: {df.columns.tolist()}")

    if time_col is None:
        raise ValueError(f"Keine Zeit-Spalte gefunden. Spalten: {df.columns.tolist()}")

    df_new = pd.DataFrame()
    df_new['Detektor'] = df[det_col].astype(str).str.replace('_GPU', '', regex=False)
    df_new['Hardware'] = hardware_name
    df_new['Keypoints'] = df[kp_col]
    df_new['Time_ms'] = df[time_col]

    if scene_col is not None:
        df_new['Scene'] = df[scene_col]

    if img_col is not None:
        df_new['Image_Index'] = df[img_col]

    return df_new


df_cpu_std = standardize_extraction_df(df_cpu, 'CPU')
df_gpu_std = standardize_extraction_df(df_gpu, 'GPU')

df_all = pd.concat([df_cpu_std, df_gpu_std], ignore_index=True)

# Speichern
df_all.to_csv(combined_csv_path, index=False)

print("\nCombined-CSV gespeichert:")
print(combined_csv_path)

print("\nAnzahl Zeilen:", len(df_all))
print("Spalten:", df_all.columns.tolist())
print("Detektoren:", sorted(df_all['Detektor'].dropna().unique()))
print("Hardware:", sorted(df_all['Hardware'].dropna().unique()))

display(df_all.head())

In [ ]:
import pandas as pd

combined_csv_path = os.path.join(OUTPUT_DIR, 'ganz_bild_extraction_cpu_gpu_combined.csv')

print('Lade ganz_bild_extraction_cpu_gpu_combined.csv')

df_all = pd.read_csv(combined_csv_path)

print('Datei geladen')
print('Anzahl Zeilen:', len(df_all))
print('Spalten:', df_all.columns.tolist())

display(df_all.head())

In [ ]:
# Diagramm 1
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

det_order = ['SIFT', 'ORB', 'BRISK', 'SuperPoint', 'DISK', 'KeyNetHardNet']

df_plot_kps = df_all.groupby(['Detektor', 'Hardware'])['Keypoints'].mean().reset_index()

pivot_kps = df_plot_kps.pivot(index='Detektor', columns='Hardware', values='Keypoints')
pivot_kps = pivot_kps.reindex(det_order)

x = np.arange(len(det_order))
width = 0.35

cpu_vals = pivot_kps['CPU'].fillna(0).values if 'CPU' in pivot_kps.columns else np.zeros(len(det_order))
gpu_vals = pivot_kps['GPU'].fillna(0).values if 'GPU' in pivot_kps.columns else np.zeros(len(det_order))

fig, ax = plt.subplots(figsize=(12, 7))

bars1 = ax.bar(x - width/2, cpu_vals, width, label='CPU')
bars2 = ax.bar(x + width/2, gpu_vals, width, label='GPU')

max_val = max(
    np.max(cpu_vals) if len(cpu_vals) > 0 else 0,
    np.max(gpu_vals) if len(gpu_vals) > 0 else 0
)

if max_val == 0:
    max_val = 1

ax.set_ylim(0, max_val * 1.12)

for bar, val in zip(bars1, cpu_vals):
    if val > 0:
        ax.text(
            bar.get_x() + bar.get_width()/2,
            val,
            f'{val:.0f}',
            ha='center',
            va='bottom',
            fontsize=14,
            fontweight='bold',
            color='black'
        )

for bar, val in zip(bars2, gpu_vals):
    if val > 0:
        ax.text(
            bar.get_x() + bar.get_width()/2,
            val,
            f'{val:.0f}',
            ha='center',
            va='bottom',
            fontsize=14,
            fontweight='bold',
            color='black'
        )

ax.set_xticks(x)
ax.set_xticklabels(det_order)
ax.set_title('Durchschnittliche Anzahl Keypoints: klassische Modelle und CNN-Modelle (CPU vs GPU)')
ax.set_ylabel('Ø Keypoints')
ax.set_xlabel('Extraktionsmethoden')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(
    os.path.join(OUTPUT_DIR, 'vergleich_keypoints_cpu_gpu_alle_modelle.png'),
    dpi=300
)
plt.show()

In [ ]:
# Diagramm 2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

det_order = ['SIFT', 'ORB', 'BRISK', 'SuperPoint', 'DISK', 'KeyNetHardNet']

df_plot_time = df_all.groupby(['Detektor', 'Hardware'])['Time_ms'].mean().reset_index()

pivot_time = df_plot_time.pivot(index='Detektor', columns='Hardware', values='Time_ms')
pivot_time = pivot_time.reindex(det_order)

x = np.arange(len(det_order))
width = 0.35

cpu_vals = pivot_time['CPU'].fillna(0).values if 'CPU' in pivot_time.columns else np.zeros(len(det_order))
gpu_vals = pivot_time['GPU'].fillna(0).values if 'GPU' in pivot_time.columns else np.zeros(len(det_order))

fig, ax = plt.subplots(figsize=(12, 7))

bars1 = ax.bar(x - width/2, cpu_vals, width, label='CPU')
bars2 = ax.bar(x + width/2, gpu_vals, width, label='GPU')

max_val = max(
    np.max(cpu_vals) if len(cpu_vals) > 0 else 0,
    np.max(gpu_vals) if len(gpu_vals) > 0 else 0
)

if max_val == 0:
    max_val = 1

ax.set_ylim(0, max_val * 1.12)

for bar, val in zip(bars1, cpu_vals):
    if val > 0:
        ax.text(
            bar.get_x() + bar.get_width()/2,
            val,
            f'{val:.1f}',
            ha='center',
            va='bottom',
            fontsize=14,
            fontweight='bold',
            color='black'
        )

for bar, val in zip(bars2, gpu_vals):
    if val > 0:
        ax.text(
            bar.get_x() + bar.get_width()/2,
            val,
            f'{val:.1f}',
            ha='center',
            va='bottom',
            fontsize=14,
            fontweight='bold',
            color='black'
        )

ax.set_xticks(x)
ax.set_xticklabels(det_order)
ax.set_title('Durchschnittliche Extraktionszeit: klassische Modelle und CNN-Modelle (CPU vs GPU)')
ax.set_ylabel('Ø Zeit (ms)')
ax.set_xlabel('Extraktionsmethoden')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(
    os.path.join(OUTPUT_DIR, 'vergleich_zeit_cpu_gpu_alle_modelle.png'),
    dpi=300
)
plt.show()

In [ ]:
import pickle
import random
import time
import numpy as np
import pandas as pd
import cv2
from pathlib import Path

random.seed(42)

def tuple_to_keypoint(t):
    x, y, size, angle, response, octave, class_id = t
    kp = cv2.KeyPoint(
        x=float(x),
        y=float(y),
        size=float(size),
        angle=float(angle),
        response=float(response),
        octave=int(octave),
        class_id=int(class_id)
    )
    return kp


print('Lade ganz_bild_extraction_cache.pkl')

with open(os.path.join(OUTPUT_DIR, 'ganz_bild_extraction_cache.pkl'), 'rb') as f:
    det_list = pickle.load(f)
    extraction_cache = {}

    for _ in det_list:
        det_name = pickle.load(f)
        det_cache = pickle.load(f)

        extraction_cache[det_name] = {}

        for key, value in det_cache.items():
            kps_serialized, descs, ms = value
            kps = [tuple_to_keypoint(t) for t in kps_serialized] if kps_serialized else []
            extraction_cache[det_name][key] = (kps, descs, ms)

print(f' Cache geladen: {list(extraction_cache.keys())}')


def keep_best_match_per_query(matches):
    best = {}
    for m in matches:
        qi = m.queryIdx
        if qi not in best:
            best[qi] = m
        else:
            old_dist = getattr(best[qi], 'distance', float('inf'))
            new_dist = getattr(m, 'distance', float('inf'))
            if new_dist < old_dist:
                best[qi] = m
    return list(best.values())


def compute_abcd_queryidx(kps1, kps2, matches, H_gt, scene_name=''):
    if H_gt is None:
        return {'A': 0, 'B': 0, 'C': 0, 'D': 0, 'N_Matched': 0}

    threshold = THRESH_VIEWPOINT if scene_name.startswith('v_') else THRESH_ILLUMINATION

    n1 = len(kps1)
    n2 = len(kps2)

    if n1 == 0:
        return {'A': 0, 'B': 0, 'C': 0, 'D': 0, 'N_Matched': 0}

    matches_unique = keep_best_match_per_query(matches)

    valid_matches = [
        m for m in matches_unique
        if 0 <= m.queryIdx < n1 and 0 <= m.trainIdx < n2
    ]

    matched_query_indices = set(m.queryIdx for m in valid_matches)

    if n2 == 0:
        A = 0
        B = len(valid_matches)
        C = 0
        D = n1 - len(matched_query_indices)
        return {'A': A, 'B': B, 'C': C, 'D': D, 'N_Matched': len(valid_matches)}

    pts1_all = np.float32([kp.pt for kp in kps1]).reshape(-1, 1, 2)
    pts1_proj_all = cv2.perspectiveTransform(pts1_all, H_gt).reshape(-1, 2)
    pts2_all = np.float32([kp.pt for kp in kps2]).reshape(-1, 2)

    A = 0
    B = 0

    for m in valid_matches:
        qi = m.queryIdx
        ti = m.trainIdx
        error = np.linalg.norm(pts1_proj_all[qi] - pts2_all[ti])
        if error < threshold:
            A += 1
        else:
            B += 1

    C = 0
    D = 0

    for qi in range(n1):
        if qi in matched_query_indices:
            continue

        errors = np.linalg.norm(pts2_all - pts1_proj_all[qi], axis=1)
        if np.any(errors < threshold):
            C += 1
        else:
            D += 1

    return {'A': A, 'B': B, 'C': C, 'D': D, 'N_Matched': len(valid_matches)}


def run_matching_experiment(scenes_list):
    results = []
    total = len(DETECTORS) * len(MATCHERS)
    kombo = 0

    print(f'\n{"="*90}')
    print(f'  Matching Experiment – alle Kombinationen')
    print(f'  Ausgabe nach jeder Kombination: Gesamt + A/B/C/D + Zeiten')
    print(f'{"="*90}')

    for det_name, det_fn in DETECTORS.items():
        is_binary = det_name in BINARY_DETECTORS
        cache = extraction_cache[det_name]

        print(f'\n{det_name}: {len(cache)} Bilder im Cache')

        for mat_name, mat_fn in MATCHERS.items():
            kombo += 1

            total_allg_kp = 0
            total_matched = 0
            total_A = 0
            total_B = 0
            total_C = 0
            total_D = 0
            total_det_ms = 0.0
            total_mat_ms = 0.0
            pair_count = 0

            for scene in scenes_list:
                s_path = Path(DATASET_PATH) / scene

                if (scene, 1) not in cache:
                    continue

                kps1, desc1, det_time1 = cache[(scene, 1)]
                if desc1 is None or len(kps1) == 0:
                    continue

                for img_idx in range(2, 7):
                    if (scene, img_idx) not in cache:
                        continue

                    kps2, desc2, det_time2 = cache[(scene, img_idx)]
                    if desc2 is None or len(kps2) == 0:
                        continue

                    t0 = time.time()
                    try:
                        matches = mat_fn(desc1, desc2, binary=is_binary)
                    except:
                        matches = []

                    mat_time = (time.time() - t0) * 1000

                    H_gt = load_homography(str(s_path), img_idx)
                    if H_gt is None:
                        continue

                    abcd = compute_abcd_queryidx(
                        kps1, kps2, matches, H_gt, scene_name=scene
                    )

                    total_allg_kp += len(kps1) + len(kps2)
                    total_matched += abcd['N_Matched']
                    total_A += abcd['A']
                    total_B += abcd['B']
                    total_C += abcd['C']
                    total_D += abcd['D']
                    total_det_ms += (det_time1 + det_time2) / 2
                    total_mat_ms += mat_time
                    pair_count += 1

            total_abcd = total_A + total_B + total_C + total_D

            pct_A = (100 * total_A / total_abcd) if total_abcd > 0 else 0
            pct_B = (100 * total_B / total_abcd) if total_abcd > 0 else 0
            pct_C = (100 * total_C / total_abcd) if total_abcd > 0 else 0
            pct_D = (100 * total_D / total_abcd) if total_abcd > 0 else 0

            avg_det = (total_det_ms / pair_count) if pair_count > 0 else 0
            avg_mat = (total_mat_ms / pair_count) if pair_count > 0 else 0

            print(f'\n{"-"*90}')
            print(f'[{kombo:02d}/{total}] {det_name} + {mat_name}')
            print(f'{"-"*90}')
            print(f'Allgemeine Keypoints gesamt:   {total_allg_kp}')
            print(f'Gematchte Keypoints gesamt:    {total_matched}')
            print(f'A (TP): {total_A}  ({pct_A:.2f}%)')
            print(f'B (FP): {total_B}  ({pct_B:.2f}%)')
            print(f'C (FN): {total_C}  ({pct_C:.2f}%)')
            print(f'D (TN): {total_D}  ({pct_D:.2f}%)')
            print(f'Durchschnitt Detektionszeit:   {avg_det:.2f} ms')
            print(f'Durchschnitt Matchingzeit:     {avg_mat:.2f} ms')

            results.append({
                'Detektor': det_name,
                'Matcher': mat_name,
                'Allg_Keypoints': total_allg_kp,
                'Matched_Keypoints': total_matched,
                'A': total_A,
                'B': total_B,
                'C': total_C,
                'D': total_D,
                'Pct_A': pct_A,
                'Pct_B': pct_B,
                'Pct_C': pct_C,
                'Pct_D': pct_D,
                'Det_ms': avg_det,
                'Mat_ms': avg_mat
            })

    print(f'\n{"="*90}')
    print(f'Fertig! {len(results)} Kombinationen ausgewertet.')
    print(f'{"="*90}')

    return pd.DataFrame(results)


df_matching = run_matching_experiment(scenes)

df_matching.to_csv(
    os.path.join(OUTPUT_DIR, 'matching_results_cpu.csv'),
    index=False
)

print("\n" + "="*90)
print("  TABELLE – Matching (A/B/C/D)")
print("="*90)

display(
    df_matching[
        ['Detektor', 'Matcher', 'Matched_Keypoints', 'Pct_A', 'Pct_B', 'Pct_C', 'Pct_D', 'Det_ms', 'Mat_ms']
    ].round({
        'Matched_Keypoints': 0,
        'Pct_A': 1,
        'Pct_B': 1,
        'Pct_C': 1,
        'Pct_D': 1,
        'Det_ms': 2,
        'Mat_ms': 2
    })
)

print("="*90)
print("A=TP   B=FP   C=FN   D=TN")
print("CSV gespeichert unter: outputs/matching_results_cpu.csv")

In [ ]:
df_matching.to_csv(os.path.join(OUTPUT_DIR, 'ganz_bild_matching_results.csv'), index=False)
df_matching.to_excel(os.path.join(OUTPUT_DIR, 'ganz_bild_matching_results.xlsx'), index=False)

print(f' {len(df_matching)} Kombinationen gespeichert')
print(f' Spalten: {list(df_matching.columns)}')

In [ ]:
df_matching = pd.read_csv(os.path.join(OUTPUT_DIR, 'ganz_bild_matching_results.csv'))
print(f' {len(df_matching)} Kombinationen geladen')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Metriken berechnen
df_plot = df_matching.copy()

eps = 1e-12

df_plot['Accuracy'] = (df_plot['A'] + df_plot['D']) / (
    df_plot['A'] + df_plot['B'] + df_plot['C'] + df_plot['D'] + eps
) * 100

df_plot['Precision'] = df_plot['A'] / (
    df_plot['A'] + df_plot['B'] + eps
) * 100

df_plot['Recall'] = df_plot['A'] / (
    df_plot['A'] + df_plot['C'] + eps
) * 100

df_plot['F1'] = 2 * df_plot['Precision'] * df_plot['Recall'] / (
    df_plot['Precision'] + df_plot['Recall'] + eps
)

# Reihenfolge
matcher_order = ['BruteForce', 'FLANN', 'KNN+Lowe', 'FAISS', 'CrossCheck']
detector_order = ['SIFT', 'ORB', 'BRISK', 'SuperPoint', 'DISK', 'KeyNetHardNet']

matcher_order = [m for m in matcher_order if m in df_plot['Matcher'].unique()]
detector_order = [d for d in detector_order if d in df_plot['Detektor'].unique()]

# gruppierte diagramme
def plot_grouped_metric(df, metric_col, title, ylabel, save_path=None):
    pivot = df.pivot(index='Matcher', columns='Detektor', values=metric_col)
    pivot = pivot.reindex(index=matcher_order, columns=detector_order)

    x = np.arange(len(pivot.index))
    n_detectors = len(pivot.columns)
    width = 0.12 if n_detectors >= 6 else 0.8 / max(n_detectors, 1)

    fig, ax = plt.subplots(figsize=(16, 8))

    max_val = np.nanmax(pivot.values.astype(float))

    ax.set_ylim(0, max_val * 1.30)

    y_offset = max_val * 0.02

    for i, det in enumerate(pivot.columns):
        values = pivot[det].values.astype(float)
        offsets = x + (i - (n_detectors - 1) / 2) * width
        bars = ax.bar(offsets, values, width=width, label=det)

        for bar, val in zip(bars, values):
            if pd.isna(val):
                continue

            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + y_offset,
                f'{val:.1f}',
                ha='center',
                va='bottom',
                rotation=90,
                fontsize=14,
                fontweight='bold',
                color='black',
                clip_on=True
            )

    ax.set_xticks(x)
    ax.set_xticklabels(pivot.index, rotation=0)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel('Matching-Algorithmen')
    ax.legend(title='Extraktionsvarianten')
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')

    plt.show()

# Alle zeigen
plot_grouped_metric(
    df_plot,
    metric_col='Mat_ms',
    title='Vergleich der Matching-Zeit',
    ylabel='Matching-Zeit (ms)',
    save_path=os.path.join(OUTPUT_DIR, 'ganz_bild_diagram_matching_zeit.png')
)

plot_grouped_metric(
    df_plot,
    metric_col='Pct_A',
    title='Vergleich von Pct_TP',
    ylabel='Pct_TP (%)',
    save_path=os.path.join(OUTPUT_DIR, 'ganz_bild_diagram_pct_a.png')
)

plot_grouped_metric(
    df_plot,
    metric_col='Pct_B',
    title='Vergleich von Pct_FP',
    ylabel='Pct_FP (%)',
    save_path=os.path.join(OUTPUT_DIR, 'ganz_bild_diagram_pct_b.png')
)

plot_grouped_metric(
    df_plot,
    metric_col='Pct_C',
    title='Vergleich von Pct_FN',
    ylabel='Pct_FN (%)',
    save_path=os.path.join(OUTPUT_DIR, 'ganz_bild_diagram_pct_c.png')
)

plot_grouped_metric(
    df_plot,
    metric_col='Pct_D',
    title='Vergleich von Pct_TN',
    ylabel='Pct_TN (%)',
    save_path=os.path.join(OUTPUT_DIR, 'ganz_bild_diagram_pct_d.png')
)

plot_grouped_metric(
    df_plot,
    metric_col='Accuracy',
    title='Vergleich der Accuracy',
    ylabel='Accuracy (%)',
    save_path=os.path.join(OUTPUT_DIR, 'ganz_bild_diagram_accuracy.png')
)

plot_grouped_metric(
    df_plot,
    metric_col='Precision',
    title='Vergleich der Precision',
    ylabel='Precision (%)',
    save_path=os.path.join(OUTPUT_DIR, 'ganz_bild_diagram_precision.png')
)

plot_grouped_metric(
    df_plot,
    metric_col='Recall',
    title='Vergleich des Recall',
    ylabel='Recall (%)',
    save_path=os.path.join(OUTPUT_DIR, 'ganz_bild_diagram_recall.png')
)

plot_grouped_metric(
    df_plot,
    metric_col='F1',
    title='Vergleich des F1-Score',
    ylabel='F1-Score',
    save_path=os.path.join(OUTPUT_DIR, 'ganz_bild_diagram_f1.png')
)

In [ ]:
import json
import numpy as np
from pathlib import Path

ROBOFLOW_PATH = os.path.join(DATA_DIR, 'hpatches-yolo-jpg')
IGNORE_KLASSEN = set()

def load_coco_boxes(scene, img_idx):
    coco_path = Path(ROBOFLOW_PATH) / scene / f'{scene}.coco' / 'train'
    ann_file  = coco_path / '_annotations.coco.json'

    if not ann_file.exists():
        return [], None

    with open(ann_file, 'r') as f:
        coco_data = json.load(f)

    cat_map     = {c['id']: c['name'] for c in coco_data['categories']}
    target_name = f'{img_idx}.jpg'
    img_id      = None
    img_h, img_w = None, None

    for img in coco_data['images']:
        extra_name = img.get('extra', {}).get('name', '')
        if extra_name == target_name:
            img_id = img['id']
            img_h  = img['height']
            img_w  = img['width']
            break

    if img_id is None:
        return [], None

    boxes = []
    for ann in coco_data['annotations']:
        if ann['image_id'] != img_id:
            continue
        klasse = cat_map.get(ann['category_id'], 'unknown')
        if klasse in IGNORE_KLASSEN:
            continue
        x, y, w, h = ann['bbox']
        x1 = int(float(x))
        y1 = int(float(y))
        x2 = int(float(x) + float(w))
        y2 = int(float(y) + float(h))
        boxes.append({
            'klasse': klasse,
            'x1': x1, 'y1': y1,
            'x2': x2, 'y2': y2,
        })

    return boxes, (img_h, img_w)


def scale_boxes(boxes, orig_size, target_size=IMG_SIZE):
    orig_h, orig_w = orig_size
    tgt_h,  tgt_w  = target_size
    scale_x = tgt_w / orig_w
    scale_y = tgt_h / orig_h
    scaled  = []
    for box in boxes:
        scaled.append({
            'klasse': box['klasse'],
            'x1': int(box['x1'] * scale_x),
            'y1': int(box['y1'] * scale_y),
            'x2': int(box['x2'] * scale_x),
            'y2': int(box['y2'] * scale_y),
        })
    return scaled


def get_keypoint_object_coco(kp, boxes):
    x, y = int(kp.pt[0]), int(kp.pt[1])
    for box in boxes:
        if box['x1'] <= x <= box['x2'] and \
           box['y1'] <= y <= box['y2']:
            return box['klasse']
    return 'background'


print('COCO Funktionen bereit')

In [ ]:
import pickle
import os
import numpy as np
from tqdm import tqdm

ALL_SCENES = scenes
coco_extraction_results_part1 = []
coco_cache_part1 = {}

def kps_to_list(kps):
    if not kps:
        return []
    return [(float(kp.pt[0]), float(kp.pt[1]),
             float(kp.size), float(kp.angle),
             float(kp.response), int(kp.octave),
             int(kp.class_id)) for kp in kps]

DETECTORS_PART1 = {
    'SIFT':  detect_sift,
    'ORB':   detect_orb,
    'BRISK': detect_brisk,
}

print(f'Teil 1: SIFT, ORB, BRISK')
print(f'{len(ALL_SCENES)} Szenen x 6 Bilder = {len(ALL_SCENES)*6} Bilder\n')
print(f'{"Detektor":<15} {"Bilder":>8} {"Ø KP Objekte":>13} {"Ø Zeit (ms)":>12}')
print('-' * 52)

for det_name, det_fn in DETECTORS_PART1.items():
    det_results      = []
    coco_cache_part1[det_name] = {}

    for scene in tqdm(ALL_SCENES, desc=det_name, leave=False):
        for img_idx in range(1, 7):

            img_path = Path(DATASET_PATH) / scene / f'{img_idx}.ppm'
            img      = load_image(img_path)
            if img is None:
                continue

            boxes_raw, img_size = load_coco_boxes(scene, img_idx)

            if img_size is not None and len(boxes_raw) > 0:
                orig_h, orig_w = img_size
                if orig_h != IMG_SIZE[0] or orig_w != IMG_SIZE[1]:
                    boxes = scale_boxes(boxes_raw, img_size)
                else:
                    boxes = boxes_raw
            else:
                boxes = []

            valid_boxes = []
            for box in boxes:
                x1 = max(0, box['x1'])
                y1 = max(0, box['y1'])
                x2 = min(IMG_SIZE[1], box['x2'])
                y2 = min(IMG_SIZE[0], box['y2'])
                if x2 > x1 and y2 > y1:
                    valid_boxes.append({
                        'klasse': box['klasse'],
                        'x1': x1, 'y1': y1,
                        'x2': x2, 'y2': y2,
                    })
            boxes = valid_boxes

            if len(boxes) == 0:
                t0 = time.time()
                kps_obj, descs_obj = det_fn(img)
                det_time_obj = (time.time() - t0) * 1000
                fallback = True
            else:
                maske = np.zeros(img.shape[:2], dtype=np.uint8)
                for box in boxes:
                    maske[box['y1']:box['y2'],
                          box['x1']:box['x2']] = 255
                img_masked   = cv2.bitwise_and(img, img, mask=maske)
                t0           = time.time()
                kps_obj, descs_obj = det_fn(img_masked)
                det_time_obj = (time.time() - t0) * 1000
                fallback     = False

                kps_filtered = []
                desc_idx     = []
                for i, kp in enumerate(kps_obj):
                    x, y = int(kp.pt[0]), int(kp.pt[1])
                    for box in boxes:
                        if box['x1'] <= x <= box['x2'] and \
                           box['y1'] <= y <= box['y2']:
                            kps_filtered.append(kp)
                            desc_idx.append(i)
                            break
                kps_obj   = kps_filtered
                descs_obj = descs_obj[desc_idx] \
                            if descs_obj is not None and \
                            len(desc_idx) > 0 else None

            n_kps_obj = len(kps_obj) if kps_obj else 0

            coco_cache_part1[det_name][(scene, img_idx)] = {
                'kps_obj':       kps_to_list(kps_obj),
                'descs_obj':     descs_obj,
                'boxes':         boxes,
                'det_time_obj':  round(det_time_obj, 2),
                'fallback':      fallback,
                'n_boxes':       len(boxes),
            }

            det_results.append({
                'Detektor':        det_name,
                'Typ':             'Klassisch',
                'Szene':           scene,
                'Szenentyp':       'Illumination' if scene.startswith('i_')
                                   else 'Viewpoint',
                'Bild':            img_idx,
                'KP_Nur_Objekte':  n_kps_obj,
                'Det_Zeit_Objekte':round(det_time_obj, 2),
                'N_Boxes':         len(boxes),
                'Fallback':        fallback,
            })

    coco_extraction_results_part1.extend(det_results)

    n      = len(det_results)
    avg_ko = sum(r['KP_Nur_Objekte']   for r in det_results) / n
    avg_to = sum(r['Det_Zeit_Objekte'] for r in det_results) / n
    print(f'{det_name:<15} {n:>8} {avg_ko:>13.0f} {avg_to:>12.1f}')

print('-' * 52)

# Speichern
print('\nSpeichere Teil 1')

with open(os.path.join(OUTPUT_DIR, 'coco_cache_part1.pkl'), 'wb') as f:
    pickle.dump(coco_cache_part1, f)

df_part1 = pd.DataFrame(coco_extraction_results_part1)
df_part1.to_csv(
    os.path.join(OUTPUT_DIR, 'coco_extraction_part1.csv'),
    index=False
)

groesse = os.path.getsize(
    os.path.join(OUTPUT_DIR, 'coco_cache_part1.pkl')
) / 1024 / 1024

print(f'Teil 1 gespeichert Größe: {groesse:.1f} MB')
print('SIFT, ORB, BRISK fertig')

In [ ]:
import pickle
import os
import numpy as np
from tqdm import tqdm

ALL_SCENES = scenes
coco_extraction_results_part2 = []
coco_cache_part2 = {}

DETECTORS_PART2 = {
    'SuperPoint':    detect_superpoint,
    'DISK':          detect_disk,
    'KeyNetHardNet': detect_keynethardnet,
}

print(f'Teil 2: SuperPoint, DISK, KeyNetHardNet')
print(f'{len(ALL_SCENES)} Szenen x 6 Bilder = {len(ALL_SCENES)*6} Bilder\n')
print(f'{"Detektor":<15} {"Bilder":>8} {"Ø KP Objekte":>13} {"Ø Zeit (ms)":>12}')
print('-' * 52)

for det_name, det_fn in DETECTORS_PART2.items():
    det_results      = []
    coco_cache_part2[det_name] = {}

    for scene in tqdm(ALL_SCENES, desc=det_name, leave=False):
        for img_idx in range(1, 7):

            img_path = Path(DATASET_PATH) / scene / f'{img_idx}.ppm'
            img      = load_image(img_path)
            if img is None:
                continue

            boxes_raw, img_size = load_coco_boxes(scene, img_idx)

            if img_size is not None and len(boxes_raw) > 0:
                orig_h, orig_w = img_size
                if orig_h != IMG_SIZE[0] or orig_w != IMG_SIZE[1]:
                    boxes = scale_boxes(boxes_raw, img_size)
                else:
                    boxes = boxes_raw
            else:
                boxes = []

            valid_boxes = []
            for box in boxes:
                x1 = max(0, box['x1'])
                y1 = max(0, box['y1'])
                x2 = min(IMG_SIZE[1], box['x2'])
                y2 = min(IMG_SIZE[0], box['y2'])
                if x2 > x1 and y2 > y1:
                    valid_boxes.append({
                        'klasse': box['klasse'],
                        'x1': x1, 'y1': y1,
                        'x2': x2, 'y2': y2,
                    })
            boxes = valid_boxes

            if len(boxes) == 0:
                t0 = time.time()
                kps_obj, descs_obj = det_fn(img)
                det_time_obj = (time.time() - t0) * 1000
                fallback = True
            else:
                maske = np.zeros(img.shape[:2], dtype=np.uint8)
                for box in boxes:
                    maske[box['y1']:box['y2'],
                          box['x1']:box['x2']] = 255
                img_masked   = cv2.bitwise_and(img, img, mask=maske)
                t0           = time.time()
                kps_obj, descs_obj = det_fn(img_masked)
                det_time_obj = (time.time() - t0) * 1000
                fallback     = False

                kps_filtered = []
                desc_idx     = []
                for i, kp in enumerate(kps_obj):
                    x, y = int(kp.pt[0]), int(kp.pt[1])
                    for box in boxes:
                        if box['x1'] <= x <= box['x2'] and \
                           box['y1'] <= y <= box['y2']:
                            kps_filtered.append(kp)
                            desc_idx.append(i)
                            break
                kps_obj   = kps_filtered
                descs_obj = descs_obj[desc_idx] \
                            if descs_obj is not None and \
                            len(desc_idx) > 0 else None

            n_kps_obj = len(kps_obj) if kps_obj else 0

            coco_cache_part2[det_name][(scene, img_idx)] = {
                'kps_obj':       kps_to_list(kps_obj),
                'descs_obj':     descs_obj,
                'boxes':         boxes,
                'det_time_obj':  round(det_time_obj, 2),
                'fallback':      fallback,
                'n_boxes':       len(boxes),
            }

            det_results.append({
                'Detektor':        det_name,
                'Typ':             'CNN',
                'Szene':           scene,
                'Szenentyp':       'Illumination' if scene.startswith('i_')
                                   else 'Viewpoint',
                'Bild':            img_idx,
                'KP_Nur_Objekte':  n_kps_obj,
                'Det_Zeit_Objekte':round(det_time_obj, 2),
                'N_Boxes':         len(boxes),
                'Fallback':        fallback,
            })

    coco_extraction_results_part2.extend(det_results)

    n      = len(det_results)
    avg_ko = sum(r['KP_Nur_Objekte']   for r in det_results) / n
    avg_to = sum(r['Det_Zeit_Objekte'] for r in det_results) / n
    print(f'{det_name:<15} {n:>8} {avg_ko:>13.0f} {avg_to:>12.1f}')

print('-' * 52)

# Speichern
print('\nSpeichere Teil 2')

with open(os.path.join(OUTPUT_DIR, 'coco_cache_part2.pkl'), 'wb') as f:
    pickle.dump(coco_cache_part2, f)

df_part2 = pd.DataFrame(coco_extraction_results_part2)
df_part2.to_csv(
    os.path.join(OUTPUT_DIR, 'coco_extraction_part2.csv'),
    index=False
)

groesse = os.path.getsize(
    os.path.join(OUTPUT_DIR, 'coco_cache_part2.pkl')
) / 1024 / 1024

print(f'Teil 2 gespeichert Größe: {groesse:.1f} MB')
print('SuperPoint, DISK, KeyNetHardNet fertig')

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

PATH_GANZBILD = os.path.join(OUTPUT_DIR, 'ganz_bild_extraction_cpu.csv')
PATH_PART1    = os.path.join(OUTPUT_DIR, 'coco_extraction_part1.csv')
PATH_PART2    = os.path.join(OUTPUT_DIR, 'coco_extraction_part2.csv')

for path in [PATH_GANZBILD, PATH_PART1, PATH_PART2]:
    if not os.path.exists(path):
        raise FileNotFoundError(f'Datei nicht gefunden: {path}')

# CSV laden
df_gesamt = pd.read_csv(PATH_GANZBILD)
df_part1  = pd.read_csv(PATH_PART1)
df_part2  = pd.read_csv(PATH_PART2)
df_objekt = pd.concat([df_part1, df_part2], ignore_index=True)

print(f'Ganzbild: {len(df_gesamt)} Messungen')
print(f'Objekte:  {len(df_objekt)} Messungen')

det_order = ['SIFT', 'ORB', 'BRISK', 'SuperPoint', 'DISK', 'KeyNetHardNet']

avg_gesamt = df_gesamt.groupby('Detektor').agg(
    KP=('Keypoints', 'mean'),
    Zeit=('Zeit_ms', 'mean'),
).reindex(det_order)

avg_objekt = df_objekt.groupby('Detektor').agg(
    KP=('KP_Nur_Objekte', 'mean'),
    Zeit=('Det_Zeit_Objekte', 'mean'),
).reindex(det_order)

avg_gesamt = avg_gesamt.fillna(0)
avg_objekt = avg_objekt.fillna(0)

print('\nDurchschnitt Ganzbild:')
print(avg_gesamt.round(1))

print('\nDurchschnitt Nur Annotation:')
print(avg_objekt.round(1))

# gruppierte diagramme
def plot_grouped_comparison(ax, values_left, values_right, title, ylabel):
    x = np.arange(len(det_order))
    width = 0.35

    bars1 = ax.bar(x - width/2, values_left, width, label='Ganzbild')
    bars2 = ax.bar(x + width/2, values_right, width, label='Nur Annotation')

    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel)
    ax.set_xticks(x)
    ax.set_xticklabels(det_order, rotation=25)
    ax.grid(axis='y', alpha=0.3)
    ax.legend()

    ymax = max(max(values_left), max(values_right), 1)

    for bar, val in zip(bars1, values_left):
        ax.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + ymax * 0.01,
            f'{val:.1f}',
            ha='center',
            va='bottom',
            fontsize=8
        )

    for bar, val in zip(bars2, values_right):
        ax.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + ymax * 0.01,
            f'{val:.1f}',
            ha='center',
            va='bottom',
            fontsize=8
        )

# 2 Diagramme
fig, axes = plt.subplots(2, 1, figsize=(14, 12))
fig.suptitle('Vergleich: Ganzbild-Extraction vs Nur Annotation', fontsize=16, fontweight='bold')

plot_grouped_comparison(
    axes[0],
    avg_gesamt['KP'].values,
    avg_objekt['KP'].values,
    title='Anzahl der Keypoints',
    ylabel='Ø Keypoints'
)

plot_grouped_comparison(
    axes[1],
    avg_gesamt['Zeit'].values,
    avg_objekt['Zeit'].values,
    title='Extraktionszeit',
    ylabel='Ø Zeit (ms)'
)

save_path = os.path.join(OUTPUT_DIR, 'vergleich_ganzbild_vs_annotation.png')

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(save_path, dpi=200, bbox_inches='tight')
plt.show()

print('\n2 Diagramme gespeichert:')
print(save_path)

In [ ]:
# Annotationen Hilfsfunktionen

import json
from pathlib import Path
import numpy as np

ROBOFLOW_PATH = os.path.join(DATA_DIR, 'hpatches-yolo-jpg')
IGNORE_KLASSEN = set()

def load_coco_boxes(scene, img_idx):
    coco_path = Path(ROBOFLOW_PATH) / scene / f'{scene}.coco' / 'train'
    ann_file  = coco_path / '_annotations.coco.json'

    if not ann_file.exists():
        return [], None

    with open(ann_file, 'r') as f:
        coco_data = json.load(f)

    cat_map = {c['id']: c['name'] for c in coco_data['categories']}

    target_name = f'{img_idx}.jpg'
    img_id = None
    img_h, img_w = None, None

    for img in coco_data['images']:
        extra_name = img.get('extra', {}).get('name', '')

        if extra_name == target_name:
            img_id = img['id']
            img_h  = img['height']
            img_w  = img['width']
            break

    if img_id is None:
        return [], None

    boxes = []

    for ann in coco_data['annotations']:
        if ann['image_id'] != img_id:
            continue

        klasse = cat_map.get(ann['category_id'], 'unknown')

        if klasse in IGNORE_KLASSEN:
            continue

        x, y, w, h = ann['bbox']

        x1 = int(float(x))
        y1 = int(float(y))
        x2 = int(float(x) + float(w))
        y2 = int(float(y) + float(h))

        boxes.append({
            'klasse': klasse,
            'x1': x1,
            'y1': y1,
            'x2': x2,
            'y2': y2,
        })

    return boxes, (img_h, img_w)


def scale_boxes(boxes, orig_size, target_size=IMG_SIZE):
    orig_h, orig_w = orig_size
    tgt_h, tgt_w = target_size

    scale_x = tgt_w / orig_w
    scale_y = tgt_h / orig_h

    scaled = []

    for box in boxes:
        scaled.append({
            'klasse': box['klasse'],
            'x1': int(box['x1'] * scale_x),
            'y1': int(box['y1'] * scale_y),
            'x2': int(box['x2'] * scale_x),
            'y2': int(box['y2'] * scale_y),
        })

    return scaled

print("Annotationen-Funktionen bereit")

In [ ]:
# Semantisch gefilterung expermint

import os
import gc
import json
import time
import pickle
import cv2
import numpy as np
import pandas as pd
from pathlib import Path


CACHE_PATH = os.path.join(OUTPUT_DIR, 'ganz_bild_extraction_cache.pkl')
ROBOFLOW_PATH = os.path.join(DATA_DIR, 'hpatches-yolo-jpg')

IGNORE_KLASSEN = set()


required_names = [
    'DATASET_PATH',
    'IMG_SIZE',
    'THRESH_ILLUMINATION',
    'THRESH_VIEWPOINT',
    'MATCHERS',
    'BINARY_DETECTORS',
    'load_homography'
]

for name in required_names:
    if name not in globals():
        raise RuntimeError(f"Variable/Funktion fehlt: {name}")

print("Grundvariablen vorhanden.")


def tuple_to_keypoint(t):
    x, y, size, angle, response, octave, class_id = t
    kp = cv2.KeyPoint(
        x=float(x),
        y=float(y),
        size=float(size),
        angle=float(angle),
        response=float(response),
        octave=int(octave),
        class_id=int(class_id)
    )
    return kp


def load_selected_extraction_cache(selected_detectors):
    selected_detectors = set(selected_detectors)
    selected_cache = {}

    print("Lade ausgewählte Detektoren aus Cache:", selected_detectors)

    with open(CACHE_PATH, 'rb') as f:
        det_list = pickle.load(f)

        for _ in det_list:
            det_name = pickle.load(f)
            det_cache = pickle.load(f)

            if det_name in selected_detectors:
                converted_cache = {}

                for key, value in det_cache.items():
                    kps_serialized, descs, ms = value

                    # Falls Cache als Tupel gespeichert wurde
                    if kps_serialized and isinstance(kps_serialized[0], tuple):
                        kps = [tuple_to_keypoint(t) for t in kps_serialized]
                    else:
                        kps = kps_serialized

                    converted_cache[key] = (kps, descs, ms)

                selected_cache[det_name] = converted_cache
                print(f"Geladen: {det_name} mit {len(converted_cache)} Bildern")
            else:
                del det_cache
                gc.collect()

    print("Geladene Detektoren:", list(selected_cache.keys()))
    return selected_cache


# COCO laden
def load_coco_boxes(scene, img_idx):
    coco_path = Path(ROBOFLOW_PATH) / scene / f'{scene}.coco' / 'train'
    ann_file = coco_path / '_annotations.coco.json'

    if not ann_file.exists():
        return [], None

    with open(ann_file, 'r') as f:
        coco_data = json.load(f)

    cat_map = {c['id']: c['name'] for c in coco_data['categories']}

    target_name = f'{img_idx}.jpg'
    img_id = None
    img_h, img_w = None, None

    for img in coco_data['images']:
        extra_name = img.get('extra', {}).get('name', '')

        if extra_name == target_name:
            img_id = img['id']
            img_h = img['height']
            img_w = img['width']
            break

    if img_id is None:
        return [], None

    boxes = []

    for ann in coco_data['annotations']:
        if ann['image_id'] != img_id:
            continue

        klasse = cat_map.get(ann['category_id'], 'unknown')

        if klasse in IGNORE_KLASSEN:
            continue

        x, y, w, h = ann['bbox']

        boxes.append({
            'klasse': klasse,
            'x1': int(float(x)),
            'y1': int(float(y)),
            'x2': int(float(x) + float(w)),
            'y2': int(float(y) + float(h)),
        })

    return boxes, (img_h, img_w)


def scale_boxes(boxes, orig_size, target_size=IMG_SIZE):
    orig_h, orig_w = orig_size
    tgt_h, tgt_w = target_size

    scale_x = tgt_w / orig_w
    scale_y = tgt_h / orig_h

    scaled = []

    for box in boxes:
        scaled.append({
            'klasse': box['klasse'],
            'x1': int(box['x1'] * scale_x),
            'y1': int(box['y1'] * scale_y),
            'x2': int(box['x2'] * scale_x),
            'y2': int(box['y2'] * scale_y),
        })

    return scaled


def get_valid_boxes_for_image(scene, img_idx):
    boxes_raw, img_size = load_coco_boxes(scene, img_idx)

    if img_size is not None and len(boxes_raw) > 0:
        if img_size[0] != IMG_SIZE[0] or img_size[1] != IMG_SIZE[1]:
            boxes = scale_boxes(boxes_raw, img_size)
        else:
            boxes = boxes_raw
    else:
        boxes = []

    valid_boxes = []

    for box in boxes:
        x1 = max(0, box['x1'])
        y1 = max(0, box['y1'])
        x2 = min(IMG_SIZE[1], box['x2'])
        y2 = min(IMG_SIZE[0], box['y2'])

        if x2 > x1 and y2 > y1:
            valid_boxes.append({
                'klasse': box['klasse'],
                'x1': x1,
                'y1': y1,
                'x2': x2,
                'y2': y2,
            })

    return valid_boxes


def get_keypoint_semantic_label(kp, boxes):
    x, y = int(kp.pt[0]), int(kp.pt[1])

    for box in boxes:
        if box['x1'] <= x <= box['x2'] and box['y1'] <= y <= box['y2']:
            return box['klasse']

    return 'background'


def keep_best_match_per_query(matches):
    best = {}

    for m in matches:
        qi = m.queryIdx

        if qi not in best:
            best[qi] = m
        else:
            old_dist = getattr(best[qi], 'distance', float('inf'))
            new_dist = getattr(m, 'distance', float('inf'))

            if new_dist < old_dist:
                best[qi] = m

    return list(best.values())


def filter_matches_by_semantics(kps1, kps2, matches, boxes1, boxes2):
    matches_unique = keep_best_match_per_query(matches)

    filtered_matches = []
    removed_matches = []

    for m in matches_unique:
        if not (0 <= m.queryIdx < len(kps1) and 0 <= m.trainIdx < len(kps2)):
            continue

        label1 = get_keypoint_semantic_label(kps1[m.queryIdx], boxes1)
        label2 = get_keypoint_semantic_label(kps2[m.trainIdx], boxes2)

        if label1 == label2:
            filtered_matches.append(m)
        else:
            removed_matches.append(m)

    return filtered_matches, removed_matches


def compute_abcd_after_semantic_filter(kps1, kps2, matches, H_gt, scene_name=''):
    if H_gt is None:
        return {'A': 0, 'B': 0, 'C': 0, 'D': 0, 'N_Matched': 0}

    threshold = THRESH_VIEWPOINT if scene_name.startswith('v_') else THRESH_ILLUMINATION

    n1 = len(kps1)
    n2 = len(kps2)

    if n1 == 0:
        return {'A': 0, 'B': 0, 'C': 0, 'D': 0, 'N_Matched': 0}

    valid_matches = [
        m for m in keep_best_match_per_query(matches)
        if 0 <= m.queryIdx < n1 and 0 <= m.trainIdx < n2
    ]

    matched_query_indices = set(m.queryIdx for m in valid_matches)

    if n2 == 0:
        A = 0
        B = len(valid_matches)
        C = 0
        D = n1 - len(matched_query_indices)

        return {
            'A': A,
            'B': B,
            'C': C,
            'D': D,
            'N_Matched': len(valid_matches)
        }

    pts1_all = np.float32([kp.pt for kp in kps1]).reshape(-1, 1, 2)
    pts1_proj_all = cv2.perspectiveTransform(pts1_all, H_gt).reshape(-1, 2)

    pts2_all = np.float32([kp.pt for kp in kps2]).reshape(-1, 2)

    A = 0
    B = 0

    for m in valid_matches:
        qi = m.queryIdx
        ti = m.trainIdx

        error = np.linalg.norm(pts1_proj_all[qi] - pts2_all[ti])

        if error < threshold:
            A += 1
        else:
            B += 1

    C = 0
    D = 0

    for qi in range(n1):
        if qi in matched_query_indices:
            continue

        errors = np.linalg.norm(pts2_all - pts1_proj_all[qi], axis=1)

        if np.any(errors < threshold):
            C += 1
        else:
            D += 1

    return {
        'A': A,
        'B': B,
        'C': C,
        'D': D,
        'N_Matched': len(valid_matches)
    }


def run_one_exp5_combination(det_name, mat_name, mat_fn, cache, scenes_list):
    is_binary = det_name in BINARY_DETECTORS

    total_allg_kp = 0

    total_matches_before = 0
    total_matches_after = 0
    total_removed_semantic = 0

    total_A = 0
    total_B = 0
    total_C = 0
    total_D = 0

    total_det_ms = 0.0
    total_mat_ms = 0.0
    total_filter_ms = 0.0

    pair_count = 0

    for scene in scenes_list:
        s_path = Path(DATASET_PATH) / scene

        if (scene, 1) not in cache:
            continue

        kps1, desc1, det_time1 = cache[(scene, 1)]

        if desc1 is None or len(kps1) == 0:
            continue

        boxes1 = get_valid_boxes_for_image(scene, 1)

        for img_idx in range(2, 7):
            if (scene, img_idx) not in cache:
                continue

            kps2, desc2, det_time2 = cache[(scene, img_idx)]

            if desc2 is None or len(kps2) == 0:
                continue

            boxes2 = get_valid_boxes_for_image(scene, img_idx)

            t0 = time.time()

            try:
                matches = mat_fn(desc1, desc2, binary=is_binary)
            except Exception:
                matches = []

            mat_time = (time.time() - t0) * 1000

            t1 = time.time()

            matches_filtered, matches_removed = filter_matches_by_semantics(
                kps1=kps1,
                kps2=kps2,
                matches=matches,
                boxes1=boxes1,
                boxes2=boxes2
            )

            filter_time = (time.time() - t1) * 1000

            H_gt = load_homography(str(s_path), img_idx)

            if H_gt is None:
                continue

            abcd = compute_abcd_after_semantic_filter(
                kps1=kps1,
                kps2=kps2,
                matches=matches_filtered,
                H_gt=H_gt,
                scene_name=scene
            )

            total_allg_kp += len(kps1) + len(kps2)

            total_matches_before += len(matches)
            total_matches_after += abcd['N_Matched']
            total_removed_semantic += len(matches_removed)

            total_A += abcd['A']
            total_B += abcd['B']
            total_C += abcd['C']
            total_D += abcd['D']

            total_det_ms += (det_time1 + det_time2) / 2
            total_mat_ms += mat_time
            total_filter_ms += filter_time

            pair_count += 1

    total_abcd = total_A + total_B + total_C + total_D

    pct_A = (100 * total_A / total_abcd) if total_abcd > 0 else 0
    pct_B = (100 * total_B / total_abcd) if total_abcd > 0 else 0
    pct_C = (100 * total_C / total_abcd) if total_abcd > 0 else 0
    pct_D = (100 * total_D / total_abcd) if total_abcd > 0 else 0

    accuracy = ((total_A + total_D) / total_abcd * 100) if total_abcd > 0 else 0
    precision = (total_A / (total_A + total_B) * 100) if (total_A + total_B) > 0 else 0
    recall = (total_A / (total_A + total_C) * 100) if (total_A + total_C) > 0 else 0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0

    avg_det = (total_det_ms / pair_count) if pair_count > 0 else 0
    avg_mat = (total_mat_ms / pair_count) if pair_count > 0 else 0
    avg_filter = (total_filter_ms / pair_count) if pair_count > 0 else 0

    removed_pct = (
        100 * total_removed_semantic / total_matches_before
        if total_matches_before > 0 else 0
    )

    row = {
        'Detektor': det_name,
        'Matcher': mat_name,

        'Allg_Keypoints': total_allg_kp,

        'Matches_before_semantic_filter': total_matches_before,
        'Matches_removed_by_semantic_filter': total_removed_semantic,
        'Matches_after_semantic_filter': total_matches_after,
        'Removed_semantic_pct': removed_pct,

        'A': total_A,
        'B': total_B,
        'C': total_C,
        'D': total_D,

        'Pct_A': pct_A,
        'Pct_B': pct_B,
        'Pct_C': pct_C,
        'Pct_D': pct_D,

        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1': f1,

        'Det_ms': avg_det,
        'Mat_ms': avg_mat,
        'Semantic_Filter_ms': avg_filter,
        'Pair_Count': pair_count
    }

    return row


def run_exp5_part(selected_detectors, output_prefix):
    csv_path = output_prefix + '.csv'
    xlsx_path = output_prefix + '.xlsx'
    checkpoint_path = output_prefix + '_checkpoint.pkl'

    if os.path.exists(csv_path):
        df_existing = pd.read_csv(csv_path)
        results = df_existing.to_dict('records')
        done_pairs = set(zip(df_existing['Detektor'], df_existing['Matcher']))
        print("Vorhandene Ergebnisse geladen:", csv_path)
        print("Schon fertig:", len(done_pairs), "Kombinationen")
    else:
        results = []
        done_pairs = set()

    selected_cache = load_selected_extraction_cache(selected_detectors)

    total = len(selected_detectors) * len(MATCHERS)
    counter = 0

    print("\n" + "=" * 100)
    print("Experiment 5 Teil gestartet")
    print("Detektoren:", selected_detectors)
    print("Speicherpfad:", output_prefix)
    print("=" * 100)

    for det_name in selected_detectors:
        if det_name not in selected_cache:
            print(f"Warnung: {det_name} nicht im Cache gefunden.")
            continue

        cache = selected_cache[det_name]

        for mat_name, mat_fn in MATCHERS.items():
            counter += 1

            if (det_name, mat_name) in done_pairs:
                print(f"[SKIP] {det_name} + {mat_name} ist schon gespeichert.")
                continue

            print("\n" + "-" * 100)
            print(f"[{counter}/{total}] Starte: {det_name} + {mat_name}")
            print("-" * 100)

            row = run_one_exp5_combination(
                det_name=det_name,
                mat_name=mat_name,
                mat_fn=mat_fn,
                cache=cache,
                scenes_list=scenes
            )

            results.append(row)
            done_pairs.add((det_name, mat_name))

            df_current = pd.DataFrame(results)

            df_current.to_csv(csv_path, index=False)

            with open(checkpoint_path, 'wb') as f:
                pickle.dump(results, f)

            print(f"Gespeichert nach Kombination: {det_name} + {mat_name}")
            print("CSV:", csv_path)
            print("Checkpoint:", checkpoint_path)

            print(f"A/TP: {row['A']} ({row['Pct_A']:.2f}%)")
            print(f"B/FP: {row['B']} ({row['Pct_B']:.2f}%)")
            print(f"C/FN: {row['C']} ({row['Pct_C']:.2f}%)")
            print(f"D/TN: {row['D']} ({row['Pct_D']:.2f}%)")
            print(f"Accuracy: {row['Accuracy']:.2f}%")
            print(f"Precision: {row['Precision']:.2f}%")
            print(f"Recall: {row['Recall']:.2f}%")
            print(f"F1: {row['F1']:.2f}")
            print(f"Semantisch entfernte Matches: {row['Matches_removed_by_semantic_filter']} ({row['Removed_semantic_pct']:.2f}%)")

            gc.collect()

    df_final = pd.DataFrame(results)

    df_final.to_excel(xlsx_path, index=False)

    print("\n" + "=" * 100)
    print("Teil fertig gespeichert.")
    print("CSV:", csv_path)
    print("Excel:", xlsx_path)
    print("Checkpoint:", checkpoint_path)
    print("Anzahl Ergebnisse:", len(df_final))
    print("=" * 100)

    return df_final

In [ ]:
# SIFT, ORB, BRISK

classic_detectors = ['SIFT', 'ORB', 'BRISK']

df_exp5_classic = run_exp5_part(
    selected_detectors=classic_detectors,
    output_prefix=os.path.join(OUTPUT_DIR, 'experiment5_part1_classic_semantic_filtered_geometric')
)

display(df_exp5_classic.round(2))

In [ ]:
# SuperPoint, DISK, KeyNetHardNet

cnn_detectors = ['SuperPoint', 'DISK', 'KeyNetHardNet']

df_exp5_cnn = run_exp5_part(
    selected_detectors=cnn_detectors,
    output_prefix=os.path.join(OUTPUT_DIR, 'experiment5_part2_cnn_semantic_filtered_geometric')
)

display(df_exp5_cnn.round(2))

In [ ]:
# Ergebnisse zusammenführen

import pandas as pd
import os

path_classic = os.path.join(OUTPUT_DIR, 'experiment5_part1_classic_semantic_filtered_geometric.csv')
path_cnn = os.path.join(OUTPUT_DIR, 'experiment5_part2_cnn_semantic_filtered_geometric.csv')

if not os.path.exists(path_classic):
    raise FileNotFoundError(f"Teil 1 fehlt: {path_classic}")

if not os.path.exists(path_cnn):
    raise FileNotFoundError(f"Teil 2 fehlt: {path_cnn}")

df_classic = pd.read_csv(path_classic)
df_cnn = pd.read_csv(path_cnn)

df_exp5_all = pd.concat([df_classic, df_cnn], ignore_index=True)

final_csv = os.path.join(OUTPUT_DIR, 'experiment5_all_semantic_filtered_geometric_results.csv')
final_xlsx = os.path.join(OUTPUT_DIR, 'experiment5_all_semantic_filtered_geometric_results.xlsx')

df_exp5_all.to_csv(final_csv, index=False)
df_exp5_all.to_excel(final_xlsx, index=False)

print("Experiment 5 komplett gespeichert:")
print("CSV:", final_csv)
print("Excel:", final_xlsx)
print("Anzahl Kombinationen:", len(df_exp5_all))

display(df_exp5_all.round(2))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

df_part1 = pd.read_csv(
    os.path.join(OUTPUT_DIR, 'experiment5_part1_classic_semantic_filtered_geometric.csv')
)

df_part2 = pd.read_csv(
    os.path.join(OUTPUT_DIR, 'experiment5_part2_cnn_semantic_filtered_geometric.csv')
)

df_part2 = df_part2[df_part1.columns]

# Beide Teile zusammenführen
df_sem = pd.concat([df_part1, df_part2], ignore_index=True)

# Gesamtergebnisse speichern
final_sem_path = os.path.join(OUTPUT_DIR, 'experiment5_semantic_filtered_geometric_all.csv')
df_sem.to_csv(final_sem_path, index=False)

print("Gesamtergebnisse gespeichert:")
print(final_sem_path)
print("Anzahl Zeilen:", len(df_sem))
print("Spalten:", list(df_sem.columns))

display(df_sem.round(2))


eps = 1e-12
df_plot = df_sem.copy()

if 'Semantic_Filter_ms' not in df_plot.columns:
    raise ValueError(
        "Die Spalte 'Semantic_Filter_ms' fehlt. "
    )

df_plot['Matching_plus_Semantic_ms'] = (
    df_plot['Mat_ms'] + df_plot['Semantic_Filter_ms']
)

# A=TP, B=FP, C=FN, D=TN
df_plot['Accuracy'] = (df_plot['A'] + df_plot['D']) / (
    df_plot['A'] + df_plot['B'] + df_plot['C'] + df_plot['D'] + eps
) * 100

df_plot['Precision'] = df_plot['A'] / (
    df_plot['A'] + df_plot['B'] + eps
) * 100

df_plot['Recall'] = df_plot['A'] / (
    df_plot['A'] + df_plot['C'] + eps
) * 100

df_plot['F1'] = 2 * df_plot['Precision'] * df_plot['Recall'] / (
    df_plot['Precision'] + df_plot['Recall'] + eps
)


# Reihenfolge
matcher_order = ['BruteForce', 'FLANN', 'KNN+Lowe', 'FAISS', 'CrossCheck']
detector_order = ['SIFT', 'ORB', 'BRISK', 'SuperPoint', 'DISK', 'KeyNetHardNet']

matcher_order = [m for m in matcher_order if m in df_plot['Matcher'].unique()]
detector_order = [d for d in detector_order if d in df_plot['Detektor'].unique()]


def plot_grouped_metric(df, metric_col, title, ylabel, save_path=None):

    pivot = df.groupby(['Matcher', 'Detektor'])[metric_col].mean().unstack()
    pivot = pivot.reindex(index=matcher_order, columns=detector_order)

    x = np.arange(len(pivot.index))
    n_detectors = len(pivot.columns)
    width = 0.12 if n_detectors >= 6 else 0.8 / max(n_detectors, 1)

    fig, ax = plt.subplots(figsize=(16, 8))

    max_val = np.nanmax(pivot.values.astype(float))

    if max_val == 0 or np.isnan(max_val):
        max_val = 1

    ax.set_ylim(0, max_val * 1.30)
    y_offset = max_val * 0.02

    for i, det in enumerate(pivot.columns):
        values = pivot[det].values.astype(float)
        offsets = x + (i - (n_detectors - 1) / 2) * width
        bars = ax.bar(offsets, values, width=width, label=det)

        for bar, val in zip(bars, values):
            if pd.isna(val):
                continue

            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + y_offset,
                f'{val:.1f}',
                ha='center',
                va='bottom',
                rotation=90,
                fontsize=14,
                fontweight='bold',
                color='black',
                clip_on=True
            )

    ax.set_xticks(x)
    ax.set_xticklabels(pivot.index)
    ax.set_title(title)
    ax.set_xlabel('Matching-Algorithmen')
    ax.set_ylabel(ylabel)
    ax.legend(title='Extraktionsvarianten')
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print("Gespeichert:", save_path)

    plt.show()


# 9 Diagramme
plot_grouped_metric(
    df_plot,
    metric_col='Matching_plus_Semantic_ms',
    title='geometrisch + Semantik – Vergleich der Laufzeit aus Matching und semantischer Filterung',
    ylabel='Matching-Zeit + Semantik-Filterzeit (ms)',
    save_path=os.path.join(OUTPUT_DIR, 'exp5_matching_plus_semantic_time.png')
)

plot_grouped_metric(
    df_plot,
    metric_col='Pct_A',
    title='geometrisch + Semantik – Vergleich von Pct_TP',
    ylabel='Pct_TP (%)',
    save_path=os.path.join(OUTPUT_DIR, 'exp5_pct_a.png')
)

plot_grouped_metric(
    df_plot,
    metric_col='Pct_B',
    title='geometrisch + Semantik – Vergleich von Pct_FP',
    ylabel='Pct_FP (%)',
    save_path=os.path.join(OUTPUT_DIR, 'exp5_pct_b.png')
)

plot_grouped_metric(
    df_plot,
    metric_col='Pct_C',
    title='geometrisch + Semantik – Vergleich von Pct_FN',
    ylabel='Pct_FN (%)',
    save_path=os.path.join(OUTPUT_DIR, 'exp5_pct_c.png')
)

plot_grouped_metric(
    df_plot,
    metric_col='Pct_D',
    title='geometrisch + Semantik – Vergleich von Pct_TN',
    ylabel='Pct_TN (%)',
    save_path=os.path.join(OUTPUT_DIR, 'exp5_pct_d.png')
)

plot_grouped_metric(
    df_plot,
    metric_col='Accuracy',
    title='geometrisch + Semantik – Vergleich der Accuracy',
    ylabel='Accuracy (%)',
    save_path=os.path.join(OUTPUT_DIR, 'exp5_accuracy.png')
)

plot_grouped_metric(
    df_plot,
    metric_col='Precision',
    title='geometrisch + Semantik – Vergleich der Precision',
    ylabel='Precision (%)',
    save_path=os.path.join(OUTPUT_DIR, 'exp5_precision.png')
)

plot_grouped_metric(
    df_plot,
    metric_col='Recall',
    title='geometrisch + Semantik – Vergleich des Recall',
    ylabel='Recall (%)',
    save_path=os.path.join(OUTPUT_DIR, 'exp5_recall.png')
)

plot_grouped_metric(
    df_plot,
    metric_col='F1',
    title='geometrisch + Semantik – Vergleich des F1-Score',
    ylabel='F1-Score',
    save_path=os.path.join(OUTPUT_DIR, 'exp5_f1.png')
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from matplotlib.patches import Patch


# reine geometrische ergbnisse
path_geo = os.path.join(OUTPUT_DIR, 'ganz_bild_matching_results.csv')

# semantisch und geometrische ergnisse
path_sem_part1 = os.path.join(OUTPUT_DIR, 'experiment5_part1_classic_semantic_filtered_geometric.csv')
path_sem_part2 = os.path.join(OUTPUT_DIR, 'experiment5_part2_cnn_semantic_filtered_geometric.csv')

for path in [path_geo, path_sem_part1, path_sem_part2]:
    if not os.path.exists(path):
        raise FileNotFoundError(f'Datei nicht gefunden: {path}')

df_geo = pd.read_csv(path_geo)

df_sem_part1 = pd.read_csv(path_sem_part1)
df_sem_part2 = pd.read_csv(path_sem_part2)

df_sem = pd.concat([df_sem_part1, df_sem_part2], ignore_index=True)

df_sem.to_csv(
    os.path.join(OUTPUT_DIR, 'experiment5_semantic_filtered_geometric_all.csv'),
    index=False
)

print("Geometrische Evaluation:", len(df_geo), "Zeilen")
print("Geometrisch + Semantik:", len(df_sem), "Zeilen")

In [ ]:
df1 = df_geo.copy()
df2 = df_sem.copy()

if 'Mat_ms' not in df1.columns and 'Mat_Zeit_ms' in df1.columns:
    df1['Mat_ms'] = df1['Mat_Zeit_ms']

if 'Mat_ms' not in df2.columns and 'Mat_Zeit_ms' in df2.columns:
    df2['Mat_ms'] = df2['Mat_Zeit_ms']


rename_map = {
    'N_A': 'A',
    'N_B': 'B',
    'N_C': 'C',
    'N_D': 'D',
    'Mat_Zeit_ms': 'Mat_ms'
}

df1 = df1.rename(columns=rename_map)
df2 = df2.rename(columns=rename_map)


if 'Mat_ms' not in df1.columns:
    raise ValueError("In df1 fehlt die Spalte 'Mat_ms'.")

if 'Mat_ms' not in df2.columns:
    raise ValueError("In df2 fehlt die Spalte 'Mat_ms'.")

if 'Semantic_Filter_ms' not in df2.columns:
    raise ValueError(
        "In df2 fehlt die Spalte 'Semantic_Filter_ms'."
    )


df1['Time_compare_ms'] = df1['Mat_ms']

df2['Time_compare_ms'] = df2['Mat_ms'] + df2['Semantic_Filter_ms']

df2['Mat_plus_Semantic_Filter_ms'] = df2['Time_compare_ms']


def add_metrics(df):
    eps = 1e-12
    out = df.copy()

    out['Accuracy'] = (out['A'] + out['D']) / (
        out['A'] + out['B'] + out['C'] + out['D'] + eps
    ) * 100

    out['Precision'] = out['A'] / (
        out['A'] + out['B'] + eps
    ) * 100

    out['Recall'] = out['A'] / (
        out['A'] + out['C'] + eps
    ) * 100

    out['F1'] = 2 * out['Precision'] * out['Recall'] / (
        out['Precision'] + out['Recall'] + eps
    )

    return out


df1 = add_metrics(df1)
df2 = add_metrics(df2)

print("Metriken berechnet.")
print("df1 Spalten:", df1.columns.tolist())
print("df2 Spalten:", df2.columns.tolist())

print("\nZeitvergleich:")
print("df1 Time_compare_ms = Mat_ms")
print("df2 Time_compare_ms = Mat_ms + Semantic_Filter_ms")

display(
    df2[
        [
            'Detektor',
            'Matcher',
            'Mat_ms',
            'Semantic_Filter_ms',
            'Time_compare_ms'
        ]
    ].round(2)
)

In [ ]:
matcher_order = ['BruteForce', 'FLANN', 'KNN+Lowe', 'FAISS', 'CrossCheck']
detector_order = ['SIFT', 'ORB', 'BRISK', 'SuperPoint', 'DISK', 'KeyNetHardNet']

matcher_order = [
    m for m in matcher_order
    if m in set(df1['Matcher']).union(set(df2['Matcher']))
]

detector_order = [
    d for d in detector_order
    if d in set(df1['Detektor']).union(set(df2['Detektor']))
]


def plot_compare_metric(df_exp1, df_exp2, metric_col, title, ylabel, save_path=None):
    p1 = df_exp1.groupby(['Matcher', 'Detektor'])[metric_col].mean().unstack()
    p2 = df_exp2.groupby(['Matcher', 'Detektor'])[metric_col].mean().unstack()

    p1 = p1.reindex(index=matcher_order, columns=detector_order)
    p2 = p2.reindex(index=matcher_order, columns=detector_order)

    x = np.arange(len(matcher_order))

    n_det = len(detector_order)
    group_width = 0.84
    pair_width = group_width / max(n_det, 1)
    single_bar_width = pair_width / 2.0

    fig, ax = plt.subplots(figsize=(18, 8))

    color_cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']

    max_val_1 = np.nanmax(p1.values.astype(float))
    max_val_2 = np.nanmax(p2.values.astype(float))
    max_val = max(max_val_1, max_val_2)

    if np.isnan(max_val) or max_val == 0:
        max_val = 1

    ax.set_ylim(0, max_val * 1.30)
    y_offset = max_val * 0.02

    for i, det in enumerate(detector_order):
        color = color_cycle[i % len(color_cycle)]

        vals1 = p1[det].values.astype(float)
        vals2 = p2[det].values.astype(float)

        center_offsets = x + (i - (n_det - 1) / 2) * pair_width

        pos1 = center_offsets - single_bar_width / 2
        pos2 = center_offsets + single_bar_width / 2

        # Geometrisch
        bars1 = ax.bar(
            pos1,
            vals1,
            width=single_bar_width,
            color=color
        )

        # Geometrisch + Semantik
        bars2 = ax.bar(
            pos2,
            vals2,
            width=single_bar_width,
            color=color,
            hatch='///',
            edgecolor='black',
            linewidth=0.8
        )

        
        for bar, val in zip(bars1, vals1):
            if pd.isna(val):
                continue

            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + y_offset,
                f'{val:.1f}',
                ha='center',
                va='bottom',
                rotation=90,
                fontsize=12,
                fontweight='bold',
                color='black',
                clip_on=True
            )

        for bar, val in zip(bars2, vals2):
            if pd.isna(val):
                continue

            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + y_offset,
                f'{val:.1f}',
                ha='center',
                va='bottom',
                rotation=90,
                fontsize=12,
                fontweight='bold',
                color='black',
                clip_on=True
            )

    ax.set_xticks(x)
    ax.set_xticklabels(matcher_order)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('Matching-Algorithmen')
    ax.set_ylabel(ylabel)
    ax.grid(axis='y', alpha=0.3)

    detector_handles = [
        Patch(
            facecolor=color_cycle[i % len(color_cycle)],
            label=detector_order[i]
        )
        for i in range(len(detector_order))
    ]

    exp_handles = [
        Patch(
            facecolor='white',
            edgecolor='black',
            label='Geometrisch'
        ),
        Patch(
            facecolor='white',
            edgecolor='black',
            hatch='///',
            label='Geometrisch + Semantik'
        ),
    ]

    legend1 = ax.legend(
        handles=detector_handles,
        title='Detektoren',
        loc='upper left',
        bbox_to_anchor=(1.01, 1.0)
    )

    ax.add_artist(legend1)

    ax.legend(
        handles=exp_handles,
        title='Evaluation',
        loc='upper left',
        bbox_to_anchor=(1.01, 0.45)
    )

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print("Gespeichert:", save_path)

    plt.show()

In [ ]:
# 9 Vergleichsdiagramme Geometrisch vs. Geometrisch + Semantik

plot_compare_metric(
    df1,
    df2,
    'Time_compare_ms',
    'Vergleich – Laufzeit: geometrisch vs. geometrisch + Semantik',
    'Matching-Zeit + ggf. Semantik-Filterzeit (ms)',
    os.path.join(OUTPUT_DIR, 'compare_geo_vs_sem_time_matching_plus_filter.png')
)

plot_compare_metric(
    df1,
    df2,
    'Pct_A',
    'Vergleich – Pct_TP: geometrisch vs. geometrisch + Semantik',
    'Pct_TP (%)',
    os.path.join(OUTPUT_DIR, 'compare_geo_vs_sem_pct_a.png')
)

plot_compare_metric(
    df1,
    df2,
    'Pct_B',
    'Vergleich – Pct_FP: geometrisch vs. geometrisch + Semantik',
    'Pct_FP (%)',
    os.path.join(OUTPUT_DIR, 'compare_geo_vs_sem_pct_b.png')
)

plot_compare_metric(
    df1,
    df2,
    'Pct_C',
    'Vergleich – Pct_FN: geometrisch vs. geometrisch + Semantik',
    'Pct_FN (%)',
    os.path.join(OUTPUT_DIR, 'compare_geo_vs_sem_pct_c.png')
)

plot_compare_metric(
    df1,
    df2,
    'Pct_D',
    'Vergleich – Pct_TN: geometrisch vs. geometrisch + Semantik',
    'Pct_TN (%)',
    os.path.join(OUTPUT_DIR, 'compare_geo_vs_sem_pct_d.png')
)

plot_compare_metric(
    df1,
    df2,
    'Accuracy',
    'Vergleich – Accuracy: geometrisch vs. geometrisch + Semantik',
    'Accuracy (%)',
    os.path.join(OUTPUT_DIR, 'compare_geo_vs_sem_accuracy.png')
)

plot_compare_metric(
    df1,
    df2,
    'Precision',
    'Vergleich – Precision: geometrisch vs. geometrisch + Semantik',
    'Precision (%)',
    os.path.join(OUTPUT_DIR, 'compare_geo_vs_sem_precision.png')
)

plot_compare_metric(
    df1,
    df2,
    'Recall',
    'Vergleich – Recall: geometrisch vs. geometrisch + Semantik',
    'Recall (%)',
    os.path.join(OUTPUT_DIR, 'compare_geo_vs_sem_recall.png')
)

plot_compare_metric(
    df1,
    df2,
    'F1',
    'Vergleich – F1-Score: geometrisch vs. geometrisch + Semantik',
    'F1-Score',
    os.path.join(OUTPUT_DIR, 'compare_geo_vs_sem_f1.png')
)